<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/11_gemini_grounding_caching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Needs a Google API key.** Search grounding and context caching are native Gemini features, so this notebook calls `google.genai` directly rather than routing through OpenRouter. It ships without stored outputs because it cannot run on the OpenRouter-only setup used for the rest of the course. To run it, open it in Colab (badge above), set a `GOOGLE_API_KEY` (from Google AI Studio) in the environment or Colab secrets, and run the cells.

# Module 11 — Gemini Unlocks: Search Grounding and Context Caching

> **⚡ Quick path** — this is one of four modules on the 1-hour course preview.
> See the [README's Quick path section](../README.md#-quick-path---1-hour) for the sequence.

**Part 2 of the course begins here.**

Ten modules of vendor-neutral ADK — everything running on whichever model you chose. Part 2 is honest about what you give up by staying vendor-neutral. Three Gemini-specific capabilities earn their own modules because nothing in LiteLLM-land can replicate them.

This module covers two of them:

- **Google Search grounding** — Gemini can call Google Search as a built-in tool, return answers with real citations to real URLs, and surface grounding metadata showing exactly which source backs each claim. No external search-API wrangling, no hallucinated URLs. Billed separately: ~$35 per 1,000 grounded requests.
- **Long context + context caching** — Gemini 2.5+ handles 1M-token input windows. For repeat queries over the same large document, you can **cache** the document once and get a 75-90% discount on all subsequent input tokens. This is the economics trick that makes long-context agents affordable in production.

The switch from Part 1 is mechanical: we stop using the `LiteLlm(model=...)` wrapper and pass Gemini model names directly. `model="gemini-2.5-flash"` — done. Everything else about the agent (tools, sessions, workflow agents, callbacks) stays identical.

**What you'll leave with:**
- A grounded Gemini agent that cites its sources — which you then verify in the `grounding_metadata`.
- A long-context Gemini query over a manual, with token-count reporting.
- A practical understanding of when caching pays off, and why the free tier blocks the demo.

**Running cost:** under $0.01 on Google AI Studio (or free tier if you stay in quota).

# Setup

In [ ]:
!pip install -q google-adk==2.4.0 google-genai litellm==1.91.1 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null
print("✅ Packages installed.")

## API Key — Switching to Google AI Studio

Part 1 used `OPENROUTER_API_KEY`. Part 2 uses `GOOGLE_API_KEY` from [aistudio.google.com/apikey](https://aistudio.google.com/apikey). Free tier is enough for M11–M13; paid tier unlocks context caching and higher rate limits.

Set `GOOGLE_GENAI_USE_VERTEXAI=FALSE` (default) to use Google AI Studio; `TRUE` to use Vertex AI (requires GCP billing).

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
try: sys.stderr.fileno()
except Exception: sys.stderr = open(os.devnull, "w")

GOOGLE_API_KEY = None
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
        GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
        if GOOGLE_API_KEY: print("✅ API key loaded from .env file.")
    except ImportError: pass
if not GOOGLE_API_KEY:
    from getpass import getpass
    print("💡 Get one at aistudio.google.com/apikey (free tier works).")
    GOOGLE_API_KEY = getpass("Enter your Google AI Studio API key: ")
assert GOOGLE_API_KEY, "❌ No API key."
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
print("✅ Environment ready for Gemini.")

## Imports

In [ ]:
import asyncio, uuid, logging
import nest_asyncio; nest_asyncio.apply()
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

# ADK — same as Part 1
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search   # ← new: the Gemini-only built-in
from google.genai import types

# Direct google-genai SDK for caching and raw long-context calls
from google import genai

print("✅ Imports successful.")

# Part 1 — Google Search Grounding

In Part 1, every agent we built had a knowledge cutoff. Ask it about today's news and you got a refusal or a hallucinated answer. The workaround was to write your own search tool — call an external API, pass results back.

Gemini ships this differently. `google_search` is a **built-in tool**. You import it from `google.adk.tools`, add it to your agent's `tools=` list, and every query triggers a real Google Search, returns cited results, and ADK populates a `grounding_metadata` field on the event with the source URLs.

**Cost note:** ~$35 per 1,000 grounded requests. Billed separately from tokens. Budget accordingly.

In [ ]:
# Notice: plain string model name, not LiteLlm(). This is the native path.
grounded_agent = LlmAgent(
    name="grounded_agent",
    model="gemini-2.5-flash",
    description="Answers questions with up-to-date Google Search results.",
    instruction=(
        "You answer user questions using Google Search for anything that "
        "requires current information. Cite your sources by URL in your reply."
    ),
    tools=[google_search],
)

print("✅ grounded_agent ready.")

In [ ]:
APP = "m11"
USER = "student"
session_service = InMemorySessionService()

async def chat(agent, prompt: str):
    sid = f"s-{uuid.uuid4().hex[:6]}"
    await session_service.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=APP, session_service=session_service)
    msg = types.Content(role="user", parts=[types.Part(text=prompt)])
    print(f"USER: {prompt}\n")
    async for ev in runner.run_async(user_id=USER, session_id=sid, new_message=msg):
        if ev.content and ev.content.parts:
            for p in ev.content.parts:
                if p.text and p.text.strip():
                    print(f"[{ev.author}] {p.text.strip()[:400]}")
                if p.function_call:
                    print(f"[tool_call] {p.function_call.name}({dict(p.function_call.args or {})})")
        if ev.grounding_metadata:
            chunks = getattr(ev.grounding_metadata, "grounding_chunks", None) or []
            if chunks:
                print(f"\n── Grounding metadata: {len(chunks)} source(s) ──")
                for i, chunk in enumerate(chunks[:3], 1):
                    web = getattr(chunk, "web", None)
                    if web:
                        print(f"  {i}. {getattr(web, 'title', '?')}")
                        print(f"     {getattr(web, 'uri', '?')[:100]}")

await chat(grounded_agent, "What is the capital of Slovakia and its current population?")

Two things to notice.

1. **The agent did not call `google_search` as a regular tool.** There's no `[tool_call]` line in the event stream. `google_search` is handled internally by Gemini's grounding infrastructure — it's a *built-in* tool, not a Python function ADK dispatches. You see the *result* of grounding in the final response, not the call sequence.
2. **The `grounding_metadata` block** lists the actual web sources Gemini used. These are real URLs, not hallucinations. In a production agent, you'd surface these to your users — "Source: bratislava.sk (official)" — as citations under each answer.

This is the single biggest pedagogical gap between Part 1 and Part 2. In Part 1 you'd write `async def web_search(query): ...`, call a paid search API, parse JSON, feed it back. Here, you add one import to the tools list. Gemini does the rest.

# The Mixing Gotcha — Built-in Tools Don't Combine

One sharp edge worth naming before you build on this. **Gemini's built-in tools (google_search, code_execution, Vertex AI Search) cannot coexist with other tools in the same agent.** If you mix them, the Gemini API rejects the request.

```python
# This fails: google_search is a built-in; get_weather is a regular function tool
BAD = LlmAgent(
    model="gemini-2.5-flash",
    tools=[google_search, get_weather],   # ← mixing built-in + regular
)
```

The usual workaround is to wrap each built-in tool as a sub-agent via `AgentTool` (from M02/M06). The coordinator calls `AgentTool(agent=grounded_agent)` for search questions and `AgentTool(agent=weather_agent)` for weather — each sub-agent has a focused tool set, and the coordinator composes their outputs.

An ADK-1.16+ alternative for Search specifically: `google_search` accepts `bypass_multi_tools_limit=True` on Search in ADK ≥ 1.16, which lets Search coexist with other tools. Check your ADK version before you depend on this.

# Part 2 — Long Context + Context Caching

Gemini 2.5+ models handle up to **1M input tokens** (2M on the Pro variants). You can paste a 500-page technical manual, an entire codebase, or several months of email history into a single request.

The obvious problem: 500,000 input tokens at the standard rate is expensive. Ask ten questions against the same manual → 5M input tokens → ~$10 on Flash, ~$50 on Pro.

**Context caching** fixes this. You cache the large content once; subsequent queries against the cache are billed at a 75–90% discount on the cached portion. Ten queries → one full-price insertion + nine discounted.

Two caching flavors:
- **Implicit caching** — Gemini 2.5+ auto-caches repeated prefixes. Zero code changes; 75% discount.
- **Explicit caching** — `client.caches.create(...)` with a fixed TTL. You control what's cached; 90% discount.

Below: a long-context query (no caching) to demonstrate the mechanics and token counts. Then the explicit-caching pattern shown in code — without running it, because **explicit caching requires a paid tier**. The free tier has zero storage quota.

## A Long-Context Query

In [ ]:
client = genai.Client(api_key=GOOGLE_API_KEY)

# A small manual. In production this could be 100KB+.
MANUAL = '''\
RaspiKitchen v2.3 — Technical Manual.

CHAPTER 1 — Overview.
The device is a voice-controlled kitchen assistant running on a Raspberry
Pi 5 with 8GB RAM and an ESP32-S3 microphone array.

CHAPTER 2 — Installation.
Power: USB-C PD, 45W minimum. Network: WiFi 6 or Ethernet. Mount the
microphone array at eye level, 1-2m from the workspace.

CHAPTER 3 — Voice wake word.
Default wake word is "Chef". To change, edit /etc/raspikitchen/wake.conf
and restart the raspikitchen.service.

CHAPTER 4 — Tool list.
Supported tools: timer, recipe lookup, conversion, substitution, shopping
list, thermometer polling (via BLE).

CHAPTER 5 — Privacy.
Audio is processed locally by default. Cloud fallback can be enabled via
/etc/raspikitchen/cloud.conf but is OFF by default.

CHAPTER 6 — Troubleshooting.
If the device does not respond to wake word, check microphone array
connection on GPIO pins 22-27. If voice recognition is slow, verify the
local Whisper model is loaded (ps aux | grep whisper).

CHAPTER 7 — Updating.
Run: sudo raspikitchen-update. Will pull the latest image and restart on
completion. Takes 2-5 minutes.
'''

# A single direct call — no ADK, just google-genai — so we see raw token counts.
resp = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=f"Based on this manual, what is the default wake word and how do I change it?\n\nMANUAL:\n{MANUAL}",
)
print(f"Response: {resp.text[:500]}")
print(f"\nToken usage:")
print(f"  input:  {resp.usage_metadata.prompt_token_count}")
print(f"  output: {resp.usage_metadata.candidates_token_count}")
print(f"  total:  {resp.usage_metadata.total_token_count}")

Two things worth noting.

1. **The `usage_metadata` reports `prompt_token_count`** — what you paid for on the input side. For a real 100KB manual, that could be 25,000+ tokens per question. Asking ten questions = 250,000 input tokens.
2. **No caching yet.** If you ran the same query twice, you'd be billed twice for the manual. Caching is how you fix that.

## Explicit Context Caching

The API. This cell **will likely fail on the free tier** — explicit caching has a zero-quota limit. Treat it as a reference for the paid tier.

In [ ]:
# Make the cache payload larger — caches have a ~32K-token minimum.
big_manual = MANUAL * 50   # repeat the manual 50 times to hit the cache threshold
approx_tokens = len(big_manual.split()) * 1.3
print(f"Cache payload: ~{int(approx_tokens)} tokens (minimum is ~32,768)")
print()

try:
    cache = client.caches.create(
        model="gemini-2.5-flash",
        config=types.CreateCachedContentConfig(
            contents=[types.Content(role="user", parts=[types.Part(text=big_manual)])],
            system_instruction="Answer questions about the RaspiKitchen manual.",
            ttl="300s",   # 5 minutes; charged by the second
        ),
    )
    print(f"✅ Cache created: {cache.name}")
    print(f"   cached_content_token_count: {cache.usage_metadata.total_token_count}")

    # Now query against the cache. The 32K+ cached tokens are billed at ~10% of normal rate.
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="What is the default wake word?",
        config=types.GenerateContentConfig(cached_content=cache.name),
    )
    print(f"\nResponse: {resp.text[:200]}")
    print(f"Tokens — cached: {resp.usage_metadata.cached_content_token_count}, "
          f"prompt: {resp.usage_metadata.prompt_token_count}, "
          f"output: {resp.usage_metadata.candidates_token_count}")

    # Clean up — caches have a TTL but you can delete explicitly
    client.caches.delete(name=cache.name)
    print(f"✅ Cache deleted.")

except Exception as e:
    print(f"❌ Caching unavailable on this tier.")
    print(f"   Error: {str(e)[:200]}")
    print()
    print("   The free tier has TotalCachedContentStorageTokensPerModelFreeTier = 0.")
    print("   Context caching requires a paid Gemini API tier.")
    print("   The code above IS the correct pattern; it fails at the billing gate,")
    print("   not at the API surface. On a paid key, this cell completes.")

## The Economics

Gemini 2.5 Flash pricing (April 2026):

| Token type | Rate |
|---|---|
| Standard input | $0.075 per 1M tokens |
| Cached input (explicit) | ~$0.008 per 1M tokens (~90% discount) |
| Cache storage | ~$0.01 per 1M tokens per hour |
| Output | $0.30 per 1M tokens |

**Worked example.** A 100-page PDF is roughly 50K tokens. Your agent gets asked 20 questions per hour against this PDF.

- **No caching:** 20 × 50K input = 1M input tokens/hour → $0.075/hour.
- **Explicit caching:** 50K input once ($0.004) + 20 × 50K cached input ($0.008) + 1 hour × 50K storage ($0.0005) = **$0.013/hour**.

**~6x cost reduction** at 20 questions/hour. Scales better the more questions you ask.

The economics tip into caching territory at maybe 3-5 queries per document per hour. Below that, implicit caching (free, 75% discount, automatic) handles most of the benefit. Above that, explicit caching is clearly worth the extra code.

# When to Pick Gemini-Native over LiteLLM-Wrapped

An honest framing of the trade-off. You can use Gemini via LiteLLM (Part 1 style) or natively (Part 2 style). The native path unlocks Gemini-only features but gives up portability.

| Feature | Native (`model="gemini-..."`) | LiteLLM (`LiteLlm(model="openrouter/google/...")`) |
|---|---|---|
| Basic chat / tool calls | ✅ | ✅ |
| `google_search` built-in tool | ✅ | ❌ |
| `BuiltInCodeExecutor` | ✅ | ❌ |
| Context caching | ✅ | ❌ |
| `ThinkingConfig` / budgets (M12) | ✅ | ⚠️ partial via `reasoning` param |
| Live API / voice (M13) | ✅ | ❌ |
| Switch to Claude/GPT/Qwen in one line | ❌ | ✅ |

**The practical rule:** use native Gemini when you need a feature that doesn't translate through LiteLLM. For everything else, LiteLLM-wrapped keeps your code portable.

In a production system, it's common to have *both* — a LiteLLM-wrapped agent for routine queries (portable, multi-model for failover) and a native-Gemini agent for tasks that need grounding, long context, or Live API (locked to Gemini, but with capabilities nothing else provides).

# Your Turn

1. **Grounding with instructions.** Modify `grounded_agent`'s instruction to require the agent to always return at least two sources from `grounding_metadata`. Ask it a question. Does it comply?
2. **Built-in-tool mixing.** Try to add your own `get_weather` function tool alongside `google_search`. What error do you get? Refactor using `AgentTool` wrappers to make both work.
3. **Long-context limits.** Make `MANUAL` 50x longer. What's the input token count? At what point does it exceed the 1M limit?
4. **Implicit vs explicit caching.** Ask the long-context query twice in a row (without explicit caching). Does the second call have a non-zero `cached_content_token_count` in its usage metadata? (Implicit caching on the same model at short time intervals should populate this.)

# Key Takeaways

- **`google_search`** is a Gemini-only built-in tool. Add it to `tools=`; Gemini handles the search internally. Event's `grounding_metadata` exposes real citation URLs. ~$35 per 1,000 grounded requests.
- **Built-in tools don't mix** with regular tools in the same agent — use `AgentTool` sub-agents, or `bypass_multi_tools_limit=True` on Search in ADK ≥ 1.16.
- **Long context** — Gemini 2.5+ handles 1M input tokens (2M on Pro). Plenty of room for large documents.
- **Implicit caching** — automatic, 75% discount on repeated prefixes. Zero code changes.
- **Explicit caching** — `client.caches.create(...)` with TTL. 90% discount, but requires a paid tier (free tier quota is zero).
- **Native Gemini unlocks** what LiteLLM can't reach: grounding, code execution, caching, thinking budgets (M12), Live API (M13).
- **Hybrid production pattern**: LiteLLM-wrapped agents for portable routine tasks; native Gemini agents for tasks that need grounding, long context, or Live API.

# Next up — M12: Thinking Budgets

Gemini 2.5+ supports `ThinkingConfig` — a knob that trades latency for reasoning quality. Set `thinking_budget=0` for instant responses; set it to 8,000+ tokens for deep reasoning on hard math. We'll run the same problem at MIN vs HIGH and watch the answer quality change along with the latency cost.